# Qwen2.5-1.5B Feature Extraction — Colab

Clones only the code from GitHub, then symlinks your existing Drive
data/features directories — no large uploads needed.

**Prerequisites on Drive** (you already have these):
- `My Drive/DL-Project/data/` — raw HuggingFace datasets
- `My Drive/DL-Project/features/` — existing features (will be extended)
- `My Drive/DL-Project/outputs/` — existing outputs

**After running:** your Drive `features/` folder will contain the new `.pt` files.
Download them locally and replace your local `features/` directory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Change this if your project lives elsewhere in Drive ──
DRIVE_PROJECT = '/content/drive/MyDrive/DL-Project'

## Step 1 — Clone repo (code only) and link Drive data

In [ ]:
import os, subprocess

REPO = 'https://github.com/brian-w-zhang/DL-Project.git'
CODE_DIR = '/content/DL-Project'

if not os.path.exists(CODE_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, CODE_DIR], check=True)
else:
    subprocess.run(['git', '-C', CODE_DIR, 'pull'], check=True)

os.chdir(CODE_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Symlink heavy directories from Drive into the cloned repo
# so scripts can use their default relative paths unchanged
import os

for name in ['data', 'features', 'outputs']:
    link = f'{CODE_DIR}/{name}'
    target = f'{DRIVE_PROJECT}/{name}'
    if not os.path.exists(link):
        os.symlink(target, link)
        print(f'Linked {link} -> {target}')
    else:
        print(f'Already exists: {link}')

## Step 2 — Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate tqdm pandas pyarrow

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## Step 3 — Build data splits from WildGuardTrain

70/10/20 split, 90:10 imbalance, 10k total:  
train 7000 (700 harmful) · val 1000 (100 harmful) · test 2000 (200 harmful)

In [ ]:
!python make_imbalance_split.py

## Step 4 — Process WildGuardTest → test_external.csv

1699 rows · 754 harmful · 945 benign (all of WildGuardTest)

In [ ]:
!python process_wildguardtest.py

## Step 5 — Extract features: last-token pooling, all 29 layers

Covers train / val / test / test_external (~11 700 prompts, ~20–30 min on T4).

In [ ]:
!python src/extract_features.py \
    --layers $(python -c "print(' '.join(map(str, range(29))))") \
    --pool last \
    --batch_size 32 \
    --max_length 256

## Step 6 — Extract features: mean pooling, all 29 layers

In [ ]:
!python src/extract_features.py \
    --layers $(python -c "print(' '.join(map(str, range(29))))") \
    --pool mean \
    --batch_size 32 \
    --max_length 256 \
    --out_slug qwen2.5-1.5b-mean

## Step 7 — Verify outputs

In [ ]:
import torch
from pathlib import Path

for slug in ['qwen2.5-1.5b', 'qwen2.5-1.5b-mean']:
    print(f'\n── {slug} (layer 19) ──')
    for split in ['train', 'val', 'test', 'test_external']:
        pt = Path(f'features/{slug}/layer_19/X_{split}.pt')
        if pt.exists():
            x = torch.load(pt, weights_only=True)
            print(f'  {split:15s}  {tuple(x.shape)}')
        else:
            print(f'  {split:15s}  MISSING')